# 13 · Backtest con pronósticos de lluvia: del pronóstico a la probabilidad del día

> **No es una alerta ni una predicción para decidir evacuaciones.** Es un prototipo para medir cuánto de la probabilidad de un día con huaico catalogado se puede recuperar cuando, en vez de la lluvia observada, el modelo recibe la lluvia **pronosticada**. Ante una emergencia llama al 115 (INDECI).

## Pre-registro (escrito antes de mirar resultados)

**Diseño «perfect prognosis».** El modelo de probabilidad del día (regresión logística con regularización L2, C = 0,03; antecedentes de ERA5, caudal, mar e ICEN, igual que los capítulos 05 y 15) se ajusta **solo con el registro observado hasta abril de 2022** y se aplica a entradas con la lluvia pronosticada del valle del Rímac. Reglas contra fuga de información: para pronosticar el día `t` con `lead` días de antelación, los antecedentes se desplazan `lead + 1` días y la lluvia de `t`, `t-1` y `t-2` es la pronosticada con ese mismo `lead`.

**Pronósticos.** Archivo *previous runs* de Open-Meteo (lluvia horaria en las 6 celdas de PISCOp sobre el valle, sumada al día de lluvia de 7 a. m. a 7 a. m.). Principal: UK Met Office 10 km, lead 0, temporadas 2022-23 a 2025-26. Principal para leads 1 a 3: JMA GSM (única con las 4 temporadas). Secundarios: GFS y ECMWF IFS 0,25° (sus leads 1 a 3 existen desde 2024).

**Corrección.** Mapeo de cuantiles contra PISCOp, cada temporada con un ajuste sin ella; la última (diciembre 2025 a abril 2026) con el ajuste de las tres anteriores (fuera de tiempo).

**Referencias.** (i) *techo*: el mismo modelo con la lluvia observada de PISCOp; (ii) ordenar los días por los mm crudos pronosticados (de 1 y de 3 días); (iii) climatología por mes. Ordenar por mm corregidos da el mismo AP que los crudos (la corrección es monótona): se anota, no se repite.

**Hipótesis.**
- **H-A** con lead 0 el modelo con pronóstico conserva al menos el 70 % del AP del techo;
- **H-B** el AP cae al subir el lead;
- **H-C** el modelo con pronóstico supera a ordenar por mm crudos del pronóstico;
- **H-D** con unos 14 días con huaico solo se detectan diferencias grandes: casi todo lo demás queda dentro del ruido.

**Métricas.** AP (principal), AUC, Brier, log-loss, huaicos dentro del 5 % y 10 % de días con mayor probabilidad; intervalos por bootstrap de bloques de 14 días (con 4 temporadas el bootstrap por temporadas no resuelve); mundos nulos con fechas desplazadas dentro de cada temporada (2 000) en GPU/CPU.

In [ ]:
import warnings

warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd

from illapa_research import filtro, gpu
from illapa_research import pronostico_probabilidad as PP
from illapa_research import quebradas as q
from illapa_research.catalogo import CATALOGO, cargar_cuencas, cargar_todos

RAIZ = Path("..") if Path("../datos").exists() else Path(".")
print("dispositivo:", gpu.dispositivo())
faltan = [c for c in ("ukmo", "jma", "gfs", "ecmwf") if not list((RAIZ / "datos" / "pronosticos").glob(f"{c}_*.json"))]
if faltan:  # 16 peticiones con pausa de 3 s; guarda un JSON por modelo y temporada
    PP.descargar(RAIZ / "datos" / "pronosticos")
R = PP.correr_backtest(RAIZ)
print("alineación con PISCOp (días):", R["desfases"])
print({k: m.info for k, m in R["modelos"].items()})

## 1. Cobertura de cada archivo de pronósticos

Días de temporada con pronóstico, por archivo, lead y temporada. Es lo que limita el backtest: el archivo de *previous runs* de UKMO solo tiene leads 1 a 3 desde 2025, y GFS e IFS desde 2024.

In [ ]:
cob = []
for clave in PP.MODELOS:
    rutas = sorted((RAIZ / "datos" / "pronosticos").glob(f"{clave}_*.json"))
    for lead in (0, 1, 2, 3):
        s = PP.serie_lead(rutas, lead)
        fila = {"archivo": clave, "lead": lead}
        for t, (a, b) in PP.TEMPORADAS_PRONOSTICO.items():
            fila[f"{t - 1}-{t}"] = int(s[(s.index >= a) & (s.index <= b)].notna().sum())
        cob.append(fila)
pd.DataFrame(cob)

## 2. Resultado principal: probabilidad con el pronóstico frente a sus referencias

`AP_pron`: el modelo con el pronóstico corregido. `AP_techo`: el mismo modelo con la lluvia observada, en los mismos días (hasta diciembre de 2025). `AP_mm3d_crudo`: ordenar por los mm pronosticados en 3 días. `AP_clima`: tasa del mes. La base es la fracción de días con huaico.

In [ ]:
T = PP.tabla_backtest(R, n_boot=2000, n_nulas=2000)
cols = ["archivo", "lead", "dias", "huaicos", "base", "AP_pron", "AUC_pron", "AP_mm3d_crudo", "AP_clima", "AP_techo",
        "AP_pron_mismos_dias_techo", "huaicos_en_5pc", "huaicos_en_10pc", "p_nula", "AP_ic80", "dAP_vs_mm3d_ic80"]
T[cols].round(3)

## 3. Cuánto del techo se conserva (H-A) y cómo cae con el lead (H-B)

In [ ]:
t = T[T["AP_pron_mismos_dias_techo"].notna()].copy()
t["fraccion_del_techo"] = t["AP_pron_mismos_dias_techo"] / t["AP_techo"]
t[["archivo", "lead", "huaicos_techo", "AP_techo", "AP_pron_mismos_dias_techo", "fraccion_del_techo"]].round(3)

## 4. Calibración del lead 0: bandas de probabilidad y el «MOS ligero»

Bandas de probabilidad con los huaicos observados en cada una. Después, un recalibrado de Platt hecho temporada por temporada (cada una con un ajuste sin ella): con 3 o 4 temporadas y 12 a 14 huaicos es una comprobación, no una capa para usar.

In [ ]:
y_all, prueba, ts = R["y"], R["prueba"], R["temporada"]
for clave in ("ukmo", "jma", "gfs"):
    p = R["predicciones"][f"{clave}_lead0_pron"]
    ok = prueba & p.notna().to_numpy()
    y, pp, tt = y_all[ok].to_numpy().astype(float), p[ok].to_numpy(), ts[ok]
    pc = PP.mos_platt_por_temporada(y, pp, tt)
    m = ~np.isnan(pc)
    a, b = PP.metricas_prob(y[m], pp[m]), PP.metricas_prob(y[m], pc[m])
    print(f"{clave}: observado/predicho {y.sum() / pp.sum():.2f} | Brier del clima {np.mean((R['clima'][ok].to_numpy() - y) ** 2):.4f}")
    print(f"   sin MOS: Brier {a['brier']:.4f} logloss {a['logloss']:.4f} AP {a['ap']:.3f}  |  con MOS: Brier {b['brier']:.4f} logloss {b['logloss']:.4f} AP {b['ap']:.3f}")
    display(PP.bandas_fiabilidad(y, pp).round(3))

## 5. Potencia: qué se puede y qué no se puede detectar con ~14 huaicos

Se mide con los intervalos de bloques de la diferencia de AP entre el modelo y ordenar por los mm crudos de 3 días, y con la fracción de mundos nulos que alcanza el AP real.

In [ ]:
pot = T[["archivo", "lead", "huaicos", "AP_pron", "AP_nula_p95", "p_nula", "dAP_vs_mm3d_ic80", "P(pron mejor que mm3d)"]].copy()
pot["supera_la_nula_al_95"] = pot["AP_pron"] > pot["AP_nula_p95"]
pot["dAP_lo"], pot["dAP_hi"] = T["dAP_lo"], T["dAP_hi"]
pot["diferencia_con_mm3d_detectable"] = (pot["dAP_lo"] > 0) | (pot["dAP_hi"] < 0)
pot.round(3)

## 6. Robustez: sin la temporada con más huaicos, y la variante compacta

Sin la temporada 2022-23 (donde cayeron 10 de los 14 días), y el modelo compacto (9 entradas fijadas antes).

In [ ]:
def ap_sin(clave, lead, quitar):
    p = R["predicciones"][f"{clave}_lead{lead}_pron"]
    ok = prueba & p.notna().to_numpy() & ~np.isin(ts, quitar)
    y = y_all[ok].to_numpy().astype(float)
    return PP.metricas_prob(y, p[ok].to_numpy())["ap"] if y.sum() >= 2 else float("nan"), int(y.sum())

R_c = PP.correr_backtest(RAIZ, leads=(0,), modelos=("ukmo", "jma", "gfs"), compacto=True)
filas = []
for clave in ("ukmo", "jma", "gfs"):
    ap_all, n_all = ap_sin(clave, 0, [])
    ap_s23, n_s23 = ap_sin(clave, 0, [2023])
    p = R_c["predicciones"][f"{clave}_lead0_pron"]
    ok = R_c["prueba"] & p.notna().to_numpy()
    ap_c = PP.metricas_prob(R_c["y"][ok].to_numpy().astype(float), p[ok].to_numpy())["ap"]
    filas.append({"archivo": clave, "AP completo": ap_all, "huaicos": n_all, "AP sin 2022-23": ap_s23, "huaicos sin 2022-23": n_s23, "AP compacto": ap_c})
pd.DataFrame(filas).round(3)

## 7. Por quebrada: probabilidad del día × participación (aprendida solo con temporadas hasta abril de 2022)

In [ ]:
cu = filtro.cuencas_a_tabla(cargar_cuencas())
refs = q.cargar_referencias(CATALOGO / "quebradas_referencias.csv")
ev = cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True)
hist = filtro.asignar_historial(ev, refs, cu)
temporadas = R["D"].temporadas
rk = filtro.tabla(cu, hist, temporadas)
ubic = sorted(rk.loc[rk.evidencia == "fisica+historial", "slug"])
filas = []
for clave, lead in (("ukmo", 0), ("jma", 0), ("gfs", 0), ("jma", 1)):
    r = PP.pares_pronostico(R, clave, lead, hist.activaciones, ubic, temporadas)
    filas.append({"archivo": clave, "lead": lead, **{k: v for k, v in r.items() if k not in ("top5_participacion",)}})
display(pd.DataFrame(filas).round(3))
print("top 5 de participación aprendido con datos hasta 2022:", r["top5_participacion"])

## 8. La función para un día y su prueba de «sin fuga»

`probabilidad_desde_pronostico(fecha_emision, lead, pronostico, observaciones, modelo, pisco)` calcula la probabilidad del día `fecha_emision + lead` usando solo observaciones anteriores a la emisión. Ejemplo con el 14 de marzo de 2023 (uno de los días del huaico).

In [ ]:
F = R["D"].F
modelo0 = R["modelos"][0]
p_ukmo = PP.serie_lead(sorted((RAIZ / "datos" / "pronosticos").glob("ukmo_*.json")), 0)
cor = PP.corregir_fuera_de_temporada(p_ukmo, R["pisco"])
for emision in ("2023-03-12", "2023-03-14", "2023-03-20", "2024-01-15"):
    r = PP.probabilidad_desde_pronostico(emision, 0, cor, F, modelo0, R["pisco"])
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items() if k != "aviso"})
print(PP.AVISO)

## Lo que no se puede concluir

- Con 12 a 14 días con huaico y 3 o 4 temporadas, un AP de 0,6 y uno de 0,5 no se distinguen. Solo se detectan diferencias grandes y la distancia al azar.
- El archivo de *previous runs* da pocos leads en cada modelo: los leads 1 a 3 de UKMO, GFS e IFS tienen 2 a 4 huaicos. Solo JMA tiene las 4 temporadas en los 4 leads, y es un modelo grueso.
- El catálogo es de huaicos reportados: lo que no se reportó cuenta como «no hubo».
- Un pronóstico que acierta el día y la hora en un modelo global de 10 a 55 km no es un pronóstico de una quebrada.